# VMamba-T from scratch

This notebook constructs the authors' VMamba-T architecture with random weights and trains every backbone and classifier parameter. It does **not** load an ImageNet checkpoint or any checkpoint from another experiment.

The shared manifest supplies the labels and the exact common split: 176 training images and 45 validation images. There is no independent test set. The adjacent `vmamba_official.py` file contains the authors' selective-scan architecture; the complete data, training, evaluation, plotting, and export workflow is contained here.

**Runtime note:** without Triton, the official implementation uses its PyTorch selective-scan fallback. A full 20-epoch run can take several hours on Apple Silicon. Progress is printed and `accuracy_by_epoch.csv` is refreshed after every epoch.

In [ ]:
import csv
import json
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

project_candidates = [
    Path.home() / 'Documents' / 'Paper replication',
    Path.home() / 'Desktop' / 'Paper replication',
]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / 'data' / 'common_split_manifest.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the Paper replication project in Documents or Desktop.')

MODEL_FOLDER = PROJECT_ROOT / 'model_reproductions' / 'could_not_run_vmamba_from_scratch'
MANIFEST = PROJECT_ROOT / 'data' / 'common_split_manifest.csv'
OUTPUT_FOLDER = MODEL_FOLDER / 'results_from_scratch'
CLASS_NAMES = ['Low', 'High']
IMAGE_SIZE = 224
EPOCHS = 20
BATCH_SIZE = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

if str(MODEL_FOLDER) not in sys.path:
    sys.path.insert(0, str(MODEL_FOLDER))
from vmamba_official import vmamba_tiny_s1l8

def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

def choose_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

In [ ]:
def read_split():
    with MANIFEST.open(newline='', encoding='utf-8') as file:
        rows = list(csv.DictReader(file))
    train, validation = [], []
    for row in rows:
        folder = 'combined_high_bw' if int(row['label']) else 'combined_low_bw'
        path = MANIFEST.parent / folder / row['file']
        if not path.is_file():
            raise FileNotFoundError(path)
        sample = (path, int(row['label']), row['file'])
        (validation if row['split'] == 'validation' else train).append(sample)
    assert len(train) == 176 and len(validation) == 45
    assert not ({sample[2] for sample in train} & {sample[2] for sample in validation})
    return train, validation

class ImageDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label, filename = self.samples[index]
        image = Image.open(path).convert('L').convert('RGB')
        return self.transform(image), label, filename

def make_loaders():
    train, validation = read_split()
    generator = torch.Generator().manual_seed(42)
    train_loader = DataLoader(
        ImageDataset(train), batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, generator=generator,
    )
    validation_loader = DataLoader(
        ImageDataset(validation), batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    )
    return train_loader, validation_loader

In [ ]:
class VMambaClassifier(nn.Module):
    def __init__(self, classes=2):
        super().__init__()
        self.backbone = vmamba_tiny_s1l8(pretrained=False)
        self.backbone.classifier.head = nn.Identity()
        self.head = nn.Linear(768, classes)

    def forward(self, images):
        return self.head(self.backbone(images))

In [ ]:
def run_epoch(model, loader, loss_function, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct = 0.0, 0
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            scores = model(images)
            loss = loss_function(scores, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += int((scores.argmax(1) == labels).sum())
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, loss_function, device):
    model.eval()
    total_loss = 0.0
    actual, predicted, probabilities, filenames = [], [], [], []
    with torch.inference_mode():
        for images, labels, names in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            probs = torch.softmax(scores, dim=1)
            total_loss += loss_function(scores, labels).item() * len(labels)
            actual.extend(labels.cpu().tolist())
            predicted.extend(scores.argmax(1).cpu().tolist())
            probabilities.extend(probs.cpu().tolist())
            filenames.extend(names)
    report = classification_report(
        actual, predicted, labels=[0, 1], target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    correct = sum(a == b for a, b in zip(actual, predicted))
    metrics = {
        'validation_loss': total_loss / len(loader.dataset),
        'validation_accuracy': correct / len(loader.dataset),
        'correct_predictions': correct,
        'incorrect_predictions': len(loader.dataset) - correct,
        'prediction_count': len(loader.dataset),
        'confusion_matrix': confusion_matrix(actual, predicted, labels=[0, 1]).tolist(),
        'Low': report['Low'], 'High': report['High'],
        'macro_f1': report['macro avg']['f1-score'],
        'weighted_f1': report['weighted avg']['f1-score'],
    }
    return metrics, list(zip(filenames, actual, predicted, probabilities))

def save_evaluation(name, model, loader, loss_function, device, output_dir):
    metrics, predictions = evaluate(model, loader, loss_function, device)
    (output_dir / f'{name}_metrics.json').write_text(json.dumps(metrics, indent=2))
    with (output_dir / f'{name}_predictions.csv').open('w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['file', 'true_label', 'predicted_label', 'probability_low', 'probability_high', 'correct'])
        for filename, true, prediction, probs in predictions:
            writer.writerow([filename, true, prediction, probs[0], probs[1], true == prediction])
    return metrics

In [ ]:
def train_from_scratch(model, output_dir):
    set_seed(42)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    device = choose_device()
    model.to(device)
    train_loader, validation_loader = make_loaders()
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    history = []
    best_loss, best_accuracy, best_accuracy_loss = float('inf'), -1.0, float('inf')
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_accuracy = run_epoch(
            model, train_loader, loss_function, device, optimizer
        )
        validation_loss, validation_accuracy = run_epoch(
            model, validation_loader, loss_function, device
        )
        row = {
            'epoch': epoch, 'training_accuracy': train_accuracy,
            'validation_accuracy': validation_accuracy,
            'training_loss': train_loss, 'validation_loss': validation_loss,
        }
        history.append(row)
        print(json.dumps(row), flush=True)
        with (output_dir / 'accuracy_by_epoch.csv').open('w', newline='') as file:
            writer = csv.DictWriter(
                file, fieldnames=['epoch', 'training_accuracy', 'validation_accuracy']
            )
            writer.writeheader()
            writer.writerows(
                {key: item[key] for key in writer.fieldnames} for item in history
            )
        if validation_loss < best_loss:
            best_loss = validation_loss
            torch.save(model.state_dict(), output_dir / 'best_validation_loss.pth')
        if validation_accuracy > best_accuracy or (
            validation_accuracy == best_accuracy and validation_loss < best_accuracy_loss
        ):
            best_accuracy, best_accuracy_loss = validation_accuracy, validation_loss
            torch.save(model.state_dict(), output_dir / 'best_validation_accuracy.pth')
    torch.save(model.state_dict(), output_dir / 'final_epoch.pth')

    accuracy_rows = [
        {key: row[key] for key in ('epoch', 'training_accuracy', 'validation_accuracy')}
        for row in history
    ]
    with (output_dir / 'accuracy_by_epoch.csv').open('w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=list(accuracy_rows[0]))
        writer.writeheader()
        writer.writerows(accuracy_rows)

    epochs = [row['epoch'] for row in history]
    training_accuracy = [row['training_accuracy'] for row in history]
    validation_accuracy = [row['validation_accuracy'] for row in history]
    figure, axis = plt.subplots(figsize=(9, 5.5))
    axis.plot(epochs, training_accuracy, 'o-', linewidth=2, label='Training accuracy')
    axis.plot(epochs, validation_accuracy, 'o-', linewidth=2, label='Validation accuracy')
    for values, label in ((training_accuracy, 'Training'), (validation_accuracy, 'Validation')):
        best_index = int(np.argmax(values))
        horizontal_offset = 70 if epochs[best_index] <= 3 else 0
        axis.scatter(epochs[best_index], values[best_index], s=80, zorder=5)
        axis.annotate(
            f'{label} max: {values[best_index]:.1%}',
            (epochs[best_index], values[best_index]),
            xytext=(horizontal_offset, 12), textcoords='offset points', ha='center',
        )
    axis.set(xlabel='Epoch', ylabel='Accuracy', title='VMamba-T from-scratch accuracy',
             xlim=(1, EPOCHS), ylim=(0, 1.08))
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()
    figure.savefig(output_dir / 'accuracy_graph.png', dpi=220, bbox_inches='tight')
    if 'ipykernel' in sys.modules:
        plt.show()
    plt.close(figure)

    summary = {}
    for name, checkpoint in (
        ('final', 'final_epoch.pth'),
        ('best_loss', 'best_validation_loss.pth'),
        ('best_accuracy', 'best_validation_accuracy.pth'),
    ):
        model.load_state_dict(torch.load(output_dir / checkpoint, map_location=device, weights_only=True))
        summary[name] = save_evaluation(
            name, model, validation_loader, loss_function, device, output_dir
        )

    result_rows = [
        {
            'checkpoint': name,
            'validation_accuracy': metrics['validation_accuracy'],
            'validation_loss': metrics['validation_loss'],
            'correct_predictions': metrics['correct_predictions'],
            'incorrect_predictions': metrics['incorrect_predictions'],
            'macro_f1': metrics['macro_f1'],
            'weighted_f1': metrics['weighted_f1'],
        }
        for name, metrics in summary.items()
    ]
    with (output_dir / 'validation_results.csv').open('w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=list(result_rows[0]))
        writer.writeheader()
        writer.writerows(result_rows)

    configuration = {
        'model': 'VMamba-T', 'initialization': 'random',
        'architecture_source': 'vmamba_official.py', 'image_size': IMAGE_SIZE,
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
        'optimizer': 'AdamW', 'seed': 42,
        'training_images': 176, 'validation_images': 45, 'test_images': 0,
        'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'total_parameters': sum(p.numel() for p in model.parameters()), 'device': str(device),
    }
    (output_dir / 'run_configuration.json').write_text(json.dumps(configuration, indent=2))
    (output_dir / 'result_summary.json').write_text(json.dumps(summary, indent=2))
    print('\nValidation results:')
    for row in result_rows:
        print(f"{row['checkpoint']:>13}: {row['validation_accuracy']:.2%} "
              f"({row['correct_predictions']}/45 correct)")
    return summary

In [ ]:
set_seed(42)
model = VMambaClassifier(classes=2)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
assert total_parameters == trainable_parameters
print(f'VMamba-T initialized randomly ({total_parameters:,} trainable parameters)')

results = train_from_scratch(model, OUTPUT_FOLDER)

The `results_from_scratch` folder contains the epoch accuracy CSV, checkpoint summary CSV, per-image validation predictions, accuracy graph, three model checkpoints, and run configuration. There is no independent test split, so all held-out scores are correctly reported as **validation accuracy**.